In [7]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [8]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [9]:
import json

from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel

from gitsource import GithubRepositoryDataReader
from evaluation_utils import llm_structured


load_dotenv()

openai_client = OpenAI()

In [10]:
class Questions(BaseModel):
    questions: list[str]

In [11]:
q1_filenames = [
    "01-agentic-rag/lessons/01-intro.md",
    "01-agentic-rag/lessons/02-environment.md",
    "01-agentic-rag/lessons/03-rag.md",
]

documents_by_filename = {
    document["filename"]: document
    for document in documents
}

q1_documents = [
    documents_by_filename[filename]
    for filename in q1_filenames
]

[document["filename"] for document in q1_documents]

['01-agentic-rag/lessons/01-intro.md',
 '01-agentic-rag/lessons/02-environment.md',
 '01-agentic-rag/lessons/03-rag.md']

In [12]:
input_tokens = []
generated_records = []

for document in q1_documents:
    user_prompt = json.dumps(
        {
            "filename": document["filename"],
            "content": document["content"],
        }
    )

    result, usage = llm_structured(
        client=openai_client,
        instructions=data_gen_instructions,
        user_prompt=user_prompt,
        output_type=Questions,
    )

    input_tokens.append(usage.input_tokens)

    for question in result.questions:
        generated_records.append(
            {
                "question": question,
                "filename": document["filename"],
            }
        )

    print(document["filename"])
    print("Input tokens:", usage.input_tokens)
    print("Questions generated:", len(result.questions))
    print()

01-agentic-rag/lessons/01-intro.md
Input tokens: 1021
Questions generated: 5

01-agentic-rag/lessons/02-environment.md
Input tokens: 1287
Questions generated: 5

01-agentic-rag/lessons/03-rag.md
Input tokens: 1754
Questions generated: 5



In [13]:
print("Input tokens:", input_tokens)
print("Number of records:", len(generated_records))

Input tokens: [1021, 1287, 1754]
Number of records: 15


In [14]:
average_input_tokens = sum(input_tokens) / len(input_tokens)

print("Average input tokens:", average_input_tokens)

Average input tokens: 1354.0


In [15]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

len(chunks)

295

In [16]:
from minsearch import Index

text_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

text_index.fit(chunks)

In [17]:
def text_search(query, num_results=5):
    return text_index.search(
        query,
        num_results=num_results
    )

In [18]:
import pandas as pd

ground_truth_url = (
    "https://raw.githubusercontent.com/"
    "DataTalksClub/llm-zoomcamp/main/"
    "cohorts/2026/04-evaluation/ground-truth.csv"
)

df_ground_truth = pd.read_csv(ground_truth_url)

ground_truth = df_ground_truth.to_dict(orient="records")

In [19]:
q = ground_truth[0]["question"]

q

"What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?"

In [20]:
text_results = text_search(q)

text_results[0]["filename"]

'01-agentic-rag/lessons/03-rag.md'

In [21]:
[
    result["filename"]
    for result in text_results
]

['01-agentic-rag/lessons/03-rag.md',
 '01-agentic-rag/lessons/13-function-calling.md',
 '01-agentic-rag/lessons/03-rag.md',
 '01-agentic-rag/lessons/13-function-calling.md',
 '01-agentic-rag/lessons/01-intro.md']

In [22]:
text_results[0]["filename"]

'01-agentic-rag/lessons/03-rag.md'

In [23]:
%pip install onnxruntime tokenizers numpy

/workspaces/llm-zoomcamp-2026-code/.venv/bin/python: No module named pip


Note: you may need to restart the kernel to use updated packages.


In [24]:
import sys
import onnxruntime as ort
import tokenizers
import numpy as np

print("Python:", sys.executable)
print("ONNX Runtime:", ort.__version__)
print("Tokenizers:", tokenizers.__version__)
print("NumPy:", np.__version__)

Python: /workspaces/llm-zoomcamp-2026-code/.venv/bin/python
ONNX Runtime: 1.27.0
Tokenizers: 0.22.2
NumPy: 2.4.6


In [25]:
from pathlib import Path
import sys

project_root = Path("/workspaces/llm-zoomcamp-2026-code")
onnx_dir = project_root / "llm-zoomcamp-onnx"
model_path = onnx_dir / "models" / "Xenova" / "all-MiniLM-L6-v2"

print("ONNX directory:", onnx_dir)
print("embedder.py exists:", (onnx_dir / "embedder.py").exists())
print("Model directory exists:", model_path.exists())

if not (onnx_dir / "embedder.py").exists():
    raise FileNotFoundError(f"Missing: {onnx_dir / 'embedder.py'}")

if not model_path.exists():
    raise FileNotFoundError(f"Missing model directory: {model_path}")

if str(onnx_dir) not in sys.path:
    sys.path.insert(0, str(onnx_dir))

from embedder import Embedder

embedder = Embedder(str(model_path))

ONNX directory: /workspaces/llm-zoomcamp-2026-code/llm-zoomcamp-onnx
embedder.py exists: True
Model directory exists: True


In [26]:
test_vector = embedder.encode("This is a test")

print(test_vector.shape)

(384,)


In [27]:
chunk_texts = [chunk["content"] for chunk in chunks]

chunk_vectors = embedder.encode_batch(chunk_texts)

print(chunk_vectors.shape)

(295, 384)


In [28]:
from minsearch import VectorSearch

vector_index = VectorSearch(
    keyword_fields=["filename"]
)

vector_index.fit(chunk_vectors, chunks)

In [29]:
def vector_search(query, num_results=5):
    query_vector = embedder.encode(query)

    return vector_index.search(
        query_vector,
        num_results=num_results
    )

In [30]:
q = ground_truth[0]["question"]

vector_results = vector_search(q)

vector_results[0]["filename"]

'01-agentic-rag/lessons/01-intro.md'

In [31]:
[
    result["filename"]
    for result in vector_results
]


['01-agentic-rag/lessons/01-intro.md',
 '04-evaluation/lessons/11-evaluation-intro.md',
 '04-evaluation/lessons/12-rag-answers.md',
 '01-agentic-rag/lessons/10-rag-next-steps.md',
 '06-best-practices/lessons/01-intro.md']

In [32]:
from tqdm.auto import tqdm


def compute_relevance(q, search_function):
    expected_filename = q["filename"]

    results = search_function(
        query=q["question"]
    )

    relevance = []

    for result in results:
        is_relevant = result["filename"] == expected_filename
        relevance.append(int(is_relevant))

    return relevance


def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total


def hit_rate(relevance):
    hits = 0

    for line in relevance:
        if 1 in line:
            hits += 1

    return hits / len(relevance)


def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank, value in enumerate(line):
            if value == 1:
                total_score += 1 / (rank + 1)
                break

    return total_score / len(relevance)


def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(
        ground_truth,
        search_function,
    )

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [33]:
text_evaluation = evaluate(
    ground_truth,
    text_search,
)

text_evaluation

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.7583333333333333, 'mrr': 0.5942592592592594}

In [34]:
vector_evaluation = evaluate(
    ground_truth,
    vector_search,
)

vector_evaluation

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.725, 'mrr': 0.5486111111111112}

In [35]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])

            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(
        scores,
        key=scores.get,
        reverse=True,
    )

    return [
        docs[key]
        for key in ranked[:num_results]
    ]

In [36]:
def hybrid_search(query, k=60):
    text_results = text_search(
        query,
        num_results=10,
    )

    vector_results = vector_search(
        query,
        num_results=10,
    )

    return rrf(
        [text_results, vector_results],
        k=k,
    )

In [37]:
k_values = [1, 50, 100, 200]

In [38]:
hybrid_evaluations = {}

for k in k_values:
    result = evaluate(
        ground_truth,
        lambda query, k=k: hybrid_search(query, k=k),
    )

    hybrid_evaluations[k] = result

    print(
        f"k={k}: "
        f"hit_rate={result['hit_rate']:.4f}, "
        f"mrr={result['mrr']:.4f}"
    )

  0%|          | 0/360 [00:00<?, ?it/s]

k=1: hit_rate=0.8389, mrr=0.6482


  0%|          | 0/360 [00:00<?, ?it/s]

k=50: hit_rate=0.8361, mrr=0.6379


  0%|          | 0/360 [00:00<?, ?it/s]

k=100: hit_rate=0.8361, mrr=0.6379


  0%|          | 0/360 [00:00<?, ?it/s]

k=200: hit_rate=0.8361, mrr=0.6379


In [39]:
best_k = max(
    hybrid_evaluations,
    key=lambda k: (
        hybrid_evaluations[k]["mrr"],
        -k,
    ),
)

print("Best k:", best_k)
print("Best result:", hybrid_evaluations[best_k])

Best k: 1
Best result: {'hit_rate': 0.8388888888888889, 'mrr': 0.6481944444444449}
